# EX: Simulating Policy Iteration

In this exercise, we will program a simplified Policy Iteration loop. We return to the same 3-sector tactical grid (`Base`, `Contested`, `Target`) from the previous Value Iteration lab, but this time our AI agent starts with a fixed **playbook (policy)** instead of raw values.

The agent at `Contested` may choose to either **Push** to `Target` (reward +10, then the mission ends) or **Retreat** to `Base` (reward 0). The agent at `Base` has only one available action: move to `Contested` (reward 0). `Target` is a terminal sector.

We will automate the two-step loop of Policy Iteration to observe how the playbook itself converges to the optimal policy, often in far fewer outer iterations than Value Iteration required.

**Lab Steps:**
*   **Initialize** an arbitrary (and deliberately suboptimal) policy at `Contested`.
*   **Define the Policy Evaluation loop**, which computes $V^\pi$ for the *current fixed* policy only.
*   **Define Policy Extraction**, which performs a one-step lookahead to find a better action, if one exists.
*   **Iterate** between evaluation and extraction until the policy stops changing.

In [1]:
# Define the tactical environment variables
gamma = 0.9
threshold = 0.001

V = {'Base': 0.0, 'Contested': 0.0, 'Target': 0.0}  # warm-started across policy iterations
policy = {'Contested': 'Retreat'}  # deliberately suboptimal starting policy, pi_0


def evaluate_policy(policy, V, gamma):
    """Policy Evaluation: repeatedly apply the Bellman update for the FIXED policy
    (no max_a) until the values for that policy converge."""
    sweeps = 0
    while True:
        V_new = V.copy()

        # Base has only one action: Move to Contested (reward 0)
        V_new['Base'] = 0 + gamma * V['Contested']

        # Contested's action is dictated entirely by the current policy
        if policy['Contested'] == 'Push':
            V_new['Contested'] = 10 + gamma * V['Target']
        else:  # 'Retreat'
            V_new['Contested'] = 0 + gamma * V['Base']

        delta = max(abs(V_new[s] - V[s]) for s in V)
        V = V_new
        sweeps += 1

        if delta < threshold:
            return V, sweeps


def extract_policy(V, gamma):
    """Policy Extraction: one-step lookahead (arg max_a) at Contested to find
    the best action given the current values."""
    q_push = 10 + gamma * V['Target']
    q_retreat = 0 + gamma * V['Base']
    return 'Push' if q_push >= q_retreat else 'Retreat'


iteration = 0
while True:
    print(f"Policy Iteration {iteration}: Policy = {policy}")

    # Step 1: Policy Evaluation
    V, sweeps = evaluate_policy(policy, V, gamma)
    print(f"  Evaluation converged in {sweeps} sweep(s). V = Base={V['Base']:.3f}, Contested={V['Contested']:.3f}")

    # Step 2: Policy Extraction
    new_action = extract_policy(V, gamma)

    if new_action == policy['Contested']:
        print(f"  Extraction: best action is still '{new_action}' -> Policy unchanged. STOP.")
        break
    else:
        print(f"  Extraction: best action is '{new_action}' -> Policy changed from '{policy['Contested']}'. Continue.")
        policy['Contested'] = new_action
        iteration += 1

print("\nOptimal Policy Found!")
print(f"Final Policy: Contested -> '{policy['Contested']}', Base -> 'Move to Contested'")
print(f"Final Values: Base={V['Base']:.3f}, Contested={V['Contested']:.3f}, Target={V['Target']:.3f}")

Policy Iteration 0: Policy = {'Contested': 'Retreat'}
  Evaluation converged in 1 sweep(s). V = Base=0.000, Contested=0.000
  Extraction: best action is 'Push' -> Policy changed from 'Retreat'. Continue.
Policy Iteration 1: Policy = {'Contested': 'Push'}
  Evaluation converged in 3 sweep(s). V = Base=9.000, Contested=10.000
  Extraction: best action is still 'Push' -> Policy unchanged. STOP.

Optimal Policy Found!
Final Policy: Contested -> 'Push', Base -> 'Move to Contested'
Final Values: Base=9.000, Contested=10.000, Target=0.000


## Interpreting the Results

At `Policy Iteration 0`, the agent is handed the flawed policy "Always Retreat." Policy Evaluation instantly confirms this playbook is worthless: since retreating never earns a reward, $V^\pi$ converges to `Base=0.000, Contested=0.000` in a single sweep.

Policy Extraction then runs a one-step lookahead and discovers that pushing to `Target` yields $q_{\text{push}} = 10$, which beats $q_{\text{retreat}} = 0$. Because the extracted action differs from the current policy, the playbook is updated to "Always Push" and the outer loop continues.

At `Policy Iteration 1`, the agent evaluates its new "Always Push" policy. This takes a few more sweeps (3) to fully propagate the reward back to `Base`, converging to `Base=9.000, Contested=10.000` — the true optimal values, matching what Value Iteration found after many more iterations in the previous lab.

Policy Extraction is run one final time: it confirms "Push" is still the best action at `Contested`, so the extracted policy is identical to the current one. The algorithm halts, having found the optimal policy $\pi^*$ in just **2 outer policy-improvement steps**, even though each step internally required its own inner loop of evaluation sweeps.

This illustrates the core trade-off from the lesson: Policy Iteration does more work *per step* (a full evaluation loop), but typically needs far fewer *outer* steps than Value Iteration to reach the same optimal values and policy.